# 06 — RAG with Sources & Citations

> **ICAN CA Training — Generative AI & RAG (6 hours).** This notebook is part of a 14-notebook curriculum. All data is synthetic. Confidential client data must not be used with public APIs without engagement-letter authority. AI output must always be verified by a qualified professional.


## Learning objectives
1. Understand why **citations matter for a CA** (audit trail, working papers).
2. Display the retrieved chunk, source filename, and page in the answer.
3. Teach the system to say *"I could not find this"* instead of hallucinating.


In [ ]:
# --- Bootstrap (don't edit) ---
# Adds the project root to sys.path so we can do `from src.xxx import yyy`.
import sys, os
from pathlib import Path
ROOT = Path.cwd()
# Walk up until we find the project root (folder that contains src/)
for _ in range(4):
    if (ROOT / 'src').exists() and (ROOT / 'requirements.txt').exists():
        break
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print('Project root:', ROOT)


In [ ]:
from src.rag_utils import build_store_from_folder, rag_answer
store = build_store_from_folder('data/generated/pdf')
print('Store size:', len(store))

## 6.1 — Answer with citations and supporting evidence

In [ ]:
def answer_with_evidence(question, k=4):
    answer, hits = rag_answer(question, store, k=k, return_hits=True)
    print('Q:', question)
    print('\nANSWER (with inline citations):')
    print(answer)
    print('\nSOURCE CHUNKS (the evidence the model saw):')
    for i, h in enumerate(hits, 1):
        m = h['metadata']
        print(f"\n[{i}] {m.get('source')}  page {m.get('page','-')}  (score {h['score']:.3f})")
        snippet = h['text'][:300].replace('\n', ' ')
        print('   ', snippet, '...')
    return answer, hits

_ = answer_with_evidence('What are the financial covenants on the term loan?')

## 6.2 — "I don't know" behaviour

A question that the corpus **cannot** answer.

In [ ]:
_ = answer_with_evidence('What was the depreciation rate used in FY 1998/99?')

**Lesson.** The system prompt instructs the model to reply with exactly *"I could not find this in the provided documents."* when retrieval is weak or the answer is missing. This is *much* safer than guessing for a CA workflow.

## 6.3 — Build a working-paper-style record

In [ ]:
from datetime import datetime
import pandas as pd

def as_workpaper(question, k=4):
    answer, hits = rag_answer(question, store, k=k, return_hits=True)
    rows = [{
        'source': h['metadata'].get('source'),
        'page': h['metadata'].get('page'),
        'similarity': round(h['score'], 3),
        'snippet': h['text'][:200].replace('\n', ' ') + '...',
    } for h in hits]
    df = pd.DataFrame(rows)
    print(f'Question      : {question}')
    print(f'Asked at      : {datetime.now().isoformat(timespec="seconds")}')
    print('Answer:')
    print(answer)
    print('\nRetrieved evidence:')
    display(df)
    return df

_ = as_workpaper('What approval threshold applies to a NPR 6 lakh purchase?')

## Expected output

* Answers contain `[source: filename, p.N]` tags.
* The depreciation-FY1998 question returns the *I-could-not-find* sentence.
* Section 6.3 produces a clean evidence table you could paste into a working paper.


## Exercise

1. Compose 5 questions, of which 2 are *deliberately unanswerable*. Run them through `answer_with_evidence`. Did the system correctly refuse on the 2 unanswerable ones?
2. Modify `DEFAULT_SYSTEM` in `src/rag_utils.py` to ask the model to also list the **filename(s) it consulted** at the bottom of every answer.


## Common errors

| Symptom | Fix |
|---|---|
| Answer contains a citation that doesn't appear in the retrieved chunks | The model invented it. Lower temperature, tighten the system prompt, verify manually. |
| "I could not find this" for a question the docs *do* answer | Retrieval missed — try rephrasing or raise `k`. |


## ⚠️ Professional caution

A citation that *looks* like a workpaper reference is still **not** a workpaper reference until a human has verified the source contains the cited fact. Always inspect the evidence chunks.